# 08 · Data-quality framework

The profile (notebook 07) describes the data; this notebook **enforces**
what it should look like. The rules live in `barca.data.validate` and run
as part of every build (`barca build`). If any *error* rule fails, the
processed data is stamped as invalid and the feature stage refuses to run.

## Why not `df.dropna()`?

Because in event data, missing values are information. A quick
demonstration on the processed events:

In [1]:
import polars as pl

from barca import config, pipeline
from barca.data import clean, normalize, store, validate

pl.Config.set_tbl_rows(45)
pl.Config.set_tbl_width_chars(170)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_hide_dataframe_shape(True)

events = store.events().collect()
n = events.height
print(f"events:                               {n:>10,}")
print(f"after events.drop_nulls():            {events.drop_nulls().height:>10,}")
print(f"after drop_nulls(subset='outcome'):   {events.drop_nulls(subset='outcome').height:>10,}")
passes = events.filter(pl.col("type") == "Pass")
kept = passes.drop_nulls(subset="outcome")
print(f"passes kept by that dropna:           {kept.height:>10,} of {passes.height:,}  "
      f"(completion rate of what is left: {kept['pass_complete'].mean():.0%})")

events:                                3,140,830
after events.drop_nulls():                     0
after drop_nulls(subset='outcome'):      428,651
passes kept by that dropna:              180,230 of 881,588  (completion rate of what is left: 0%)


A blanket `drop_nulls()` deletes **every** row, because no event has every
field. Dropping on `outcome` keeps only failed passes: the "cleaned" data
would report a 0% pass completion rate. Missingness has to be judged per
event type, which is exactly what the rules below do.

## The rules

Each rule has a code, a table, a severity and, where the source has a
known harmless quirk, an explicit tolerance. They fall into five groups:

* **identity**: unique ids, contiguous indices, one fixture per pairing;
* **validity**: periods, clocks, coordinates on the pitch, xG in (0, 1),
  odds above 1;
* **completeness by type**: shot → xG, outcome and locations; pass → start
  and end; carry → end; substitution → replacement;
* **referential integrity**: event players in the matchday squad, related
  events and key passes inside the match, event teams in the match;
* **cross-source consistency**: goals rebuilt from events equal the
  official score, and every league match agrees with football-data.

In [2]:
pl.DataFrame(
    [{"code": r.code, "table": r.table, "severity": r.severity, "tolerance": r.tolerance, "check": r.description}
     for r in validate.RULES]
)

code,table,severity,tolerance,check
str,str,str,i64,str
"""E01""","""events""","""error""",0,"""event ids are unique"""
"""E02""","""events""","""error""",0,"""event index runs 1..n within every match"""
"""E03""","""events""","""error""",0,"""period is 1-5"""
"""E04""","""events""","""error""",0,"""minute/second are valid for the period"""
"""E05""","""events""","""warning""",5,"""periods never go backwards in index order"""
"""E06""","""events""","""warning""",1000,"""timestamps rarely go backwards within a period (related events can be recorded o…"
"""E07""","""events""","""error""",0,"""locations lie on the 120 x 80 pitch"""
"""E08""","""events""","""error""",0,"""every shot has an xG in (0, 1)"""
"""E09""","""events""","""error""",0,"""every shot has an outcome, a start and an end location"""


## Before cleaning: the interim tables

Running the same rules on the *interim* data (flattened but uncleaned)
shows what cleaning has to fix. The interim version of the lineup table
takes `started` from the lineup spells, as the raw data suggests.

In [3]:
raw_matches = normalize.matches(pl.read_parquet(config.INTERIM / "matches.parquet"))
raw_events = normalize.events(pipeline.scan_events(config.INTERIM))
bounds = normalize.period_bounds(raw_events)
minutes = normalize.player_minutes(pl.read_parquet(config.INTERIM / "positions.parquet"), bounds)
raw_lineups = pl.read_parquet(config.INTERIM / "lineups.parquet")
raw_appearances = raw_lineups.join(minutes, on=["match_id", "team_id", "player_id"], how="left").with_columns(
    pl.col("started").fill_null(False), pl.col("minutes").fill_null(0.0)
)
interim_tables = {
    "events": raw_events,
    "matches": raw_matches,
    "appearances": raw_appearances,
    "tactics": pl.read_parquet(config.INTERIM / "tactics.parquet"),
    "results": normalize.football_data_results(pl.read_parquet(config.INTERIM / "football_data.parquet")),
}
before = validate.run(interim_tables)
validate.summary(before).filter(pl.col("violations") > 0)

code,table,description,severity,violations,tolerance,passed
str,str,str,str,i64,i64,bool
"""E05""","""events""","""periods never go backwards in index order""","""warning""",1,5,true
"""E06""","""events""","""timestamps rarely go backwards within a period (related events can be recorded o…","""warning""",314,1000,true
"""E07""","""events""","""locations lie on the 120 x 80 pitch""","""error""",1536,0,false
"""L02""","""appearances""","""exactly 11 starters per team per match""","""error""",3,0,false
"""L04""","""appearances""","""team minutes are consistent with 9-11 players on the pitch""","""warning""",12,15,true


Two error rules fail before cleaning, and both trace back to findings
from the EDA:

* **E07, locations on the pitch**: touchline and by-line actions recorded
  up to 0.9 yards beyond the line. Cleaning rule C1 clips them to the
  pitch.
* **L02, 11 starters per team**: in three team-matches some starters'
  first lineup spell is labelled "Tactical Shift" instead of "Starting XI".
  Cleaning rule C4 takes `started` from the Starting XI event, which always
  lists exactly 11 players (rule E21).

## After cleaning: the processed tables

In [4]:
after = validate.run(pipeline.processed_tables())
summary = validate.summary(after)
print("error rules passed:", summary.filter(pl.col("severity") == "error")["passed"].all())
summary.filter((pl.col("violations") > 0) | (pl.col("code").is_in(["E07", "L02", "E22", "M05"])))

error rules passed: True


code,table,description,severity,violations,tolerance,passed
str,str,str,str,i64,i64,bool
"""E05""","""events""","""periods never go backwards in index order""","""warning""",1,5,true
"""E06""","""events""","""timestamps rarely go backwards within a period (related events can be recorded o…","""warning""",314,1000,true
"""E07""","""events""","""locations lie on the 120 x 80 pitch""","""error""",0,0,true
"""E22""","""events""","""goals reconstructed from events equal the official score""","""error""",0,0,true
"""M05""","""matches""","""every league match reconciles with football-data (date, teams, score)""","""error""",0,0,true
"""L02""","""appearances""","""exactly 11 starters per team per match""","""error""",0,0,true
"""L04""","""appearances""","""team minutes are consistent with 9-11 players on the pitch""","""warning""",12,15,true


Every error rule passes. Three warnings remain, each a documented quirk of
the source rather than a defect to fix:

* **E05**: one event (a first-half ball receipt) is indexed after the
  second half began. It is a single out-of-order record.
* **E06**: 314 of 3.1 million events have a timestamp slightly earlier than
  the previous event. They are paired events (a duel and the dribble it
  relates to) recorded in a different order from their timestamps. Order
  by `index` for sequences and by timestamp for durations.
* **L04**: in 12 of 1,738 team-matches (0.7%), a temporary injury exit
  ("Player Off/On") overlaps with a substitution in the lineup spells, so
  one player's minutes are inflated. The effect on per-90 rates is
  negligible and the warning keeps it visible.

Two checks are worth highlighting because they are strong end-to-end
guarantees:

* **E22**: goals rebuilt from shot and own-goal events equal the official
  score in all 869 matches. No goal events are missing or duplicated.
* **M05**: all 866 league matches agree with football-data on date, teams
  and score, so the two sources can be joined safely.

In [5]:
violations = [r for r in after if r.violations]
for r in violations:
    print(r.code, r.description)
    for row in r.sample[:2]:
        print("   ", row)

E05 periods never go backwards in index order
    {'match_id': 3825848, 'index': 2590, 'period': 1, 'type': 'Ball Receipt*'}
E06 timestamps rarely go backwards within a period (related events can be recorded out of order)
    {'match_id': 9575, 'index': 2082, 'period': 1, 'timestamp': '00:00:04.227'}
    {'match_id': 9581, 'index': 1863, 'period': 1, 'timestamp': '00:00:01.159'}
L04 team minutes are consistent with 9-11 players on the pitch
    {'match_id': 68329, 'team_id': 217, 'team_minutes': 1154.0666666666666, 'match_minutes': 98.95}
    {'match_id': 69144, 'team_id': 212, 'team_minutes': 1094.8666666666666, 'match_minutes': 95.38333333333333}


## Where the rules run

* `barca build` runs them after every rebuild, writes
  `reports/data_quality.md` and stamps `data/processed/_validation.json`.
  The feature stage calls `validate.require_stamp()` and stops if the gate
  failed.
* `tests/data_quality/` runs the same rules under pytest (`make test-data`),
  and CI runs it on a small sample of matches downloaded fresh from
  StatsBomb, so a change in the upstream data is caught without anyone
  opening a notebook.

In [6]:
validate.write_report(after, config.REPORTS / "data_quality.md")
print(open(config.REPORTS / "data_quality.md").read().split("## Samples")[0])

# Data quality report

| Rule | Table | Check | Severity | Violations | Tolerance | Status |
|---|---|---|---|---|---|---|
| E01 | events | event ids are unique | error | 0 | 0 | pass |
| E02 | events | event index runs 1..n within every match | error | 0 | 0 | pass |
| E03 | events | period is 1-5 | error | 0 | 0 | pass |
| E04 | events | minute/second are valid for the period | error | 0 | 0 | pass |
| E05 | events | periods never go backwards in index order | warning | 1 | 5 | pass |
| E06 | events | timestamps rarely go backwards within a period (related events can be recorded out of order) | warning | 314 | 1000 | pass |
| E07 | events | locations lie on the 120 x 80 pitch | error | 0 | 0 | pass |
| E08 | events | every shot has an xG in (0, 1) | error | 0 | 0 | pass |
| E09 | events | every shot has an outcome, a start and an end location | error | 0 | 0 | pass |
| E10 | events | every pass has a start and an end location | error | 0 | 0 | pass |
| E11 | events | every carry has 

## Findings

1. **The source is very clean.** Of 38 rules, every error rule passes after
   two small, targeted cleaning steps.
2. **Validation is type-aware**: expected fields are checked per event
   type, so structural nulls are never mistaken for defects.
3. **Cross-source checks give end-to-end confidence**: events reproduce
   every official score, and StatsBomb agrees with football-data on every
   league match.
4. **Known quirks are documented with tolerances**, not silently ignored.

Next: the reproducible cleaning pipeline and the feature store
(`09_feature_engineering`).